## Assignment 1 - DPT
- The code requires CUDA to run, which is not straightforward to set up on MacOS
- For this reason, I used Google Colab to run my experiments
- If you wish to replicate this, you will need to follow the steps in the readme.md file on GitHub to download checkpoints
- You can then run the cells below

## Set-up Cells

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q -U pip
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q tensorboardX pillow opencv-python matplotlib scikit-learn scipy

In [5]:
import os
from pathlib import Path
import re
import pandas as pd
import io

In [6]:
ROOT = Path('/content/drive/MyDrive/Knowledge-Enriched-DMI-main')

In [7]:
os.chdir(ROOT)
print(f"Changed current working directory to: {os.getcwd()}")

Changed current working directory to: /content/drive/MyDrive/Knowledge-Enriched-DMI-main


## Parameters passed as arguments in all commands
- model: this is the model name (VGG16/IR152/FaceNet64) that we're attacking
- iter_list: list of iteration numbers the attack should be carried out with
- num_ids: number of identities
- num_seeds: number of seeds
- improved_flag: flag used to toggle the use of Improved GAN model
- dist_flag: flag used to toggle the use of distributional recovery attack
- defense: the defense mechanism used while performing attack (values can be "noise" - Gaussian noise or "smooth" - label smoothing)
- noise_sigma: Gaussian noise factor (if defense is "noise")
- smooth_alpha: Label smoothing factor (if defense is "smooth")
- phase: phase name (phase1/phase2/phase3)

## Phase 1 - Iteration Budget v/s Accuracy Convergence Experiments

In [ ]:
# Attack VGG16
!python recovery.py \
    --model VGG16 \
    --iter_list 300,600,1200 \
    --num_ids 10 \
    --num_seeds 3 \
    --improved_flag \
    --phase phase1

[2026-03-01 17:54:25,400 INFO recovery.py line 68 6223] Namespace(model='VGG16', device='4,5,6,7', improved_flag=True, dist_flag=False, iter_list='300,600,1200', num_ids=10, num_seeds=3, phase='phase1', defense='none', noise_sigma=0.01, smooth_alpha=0.1, model_trained_against=None)
[2026-03-01 17:54:25,401 INFO recovery.py line 69 6223] => creating model ...
/content/drive/MyDrive/Knowledge-Enriched-DMI-main/discri.py:15: FutureWarning: `nn.init.normal` is now deprecated in favor of `nn.init.normal_`.
  init.normal(self.T, 0, 1)
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to p

In [ ]:
# Attack IR152
!python recovery.py \
    --model IR152 \
    --iter_list 300,600,1200 \
    --num_ids 10 \
    --num_seeds 3 \
    --improved_flag \
    --phase phase1

[2026-03-01 17:58:27,407 INFO recovery.py line 68 7317] Namespace(model='IR152', device='4,5,6,7', improved_flag=True, dist_flag=False, iter_list='300,600,1200', num_ids=10, num_seeds=3, phase='phase1', defense='none', noise_sigma=0.01, smooth_alpha=0.1, model_trained_against=None)
[2026-03-01 17:58:27,407 INFO recovery.py line 69 7317] => creating model ...
/content/drive/MyDrive/Knowledge-Enriched-DMI-main/discri.py:15: FutureWarning: `nn.init.normal` is now deprecated in favor of `nn.init.normal_`.
  init.normal(self.T, 0, 1)
[2026-03-01 17:58:40,323 INFO recovery.py line 172 7317] => Running phase1
[2026-03-01 17:58:40,327 INFO recovery.py line 191 7317] ==> Running with iter_times=300
Iteration:300	Prior Loss:0.05	Iden Loss:0.00	Attack Acc:0.80
Time:29.00	Acc:0.80	
Iteration:300	Prior Loss:0.06	Iden Loss:0.00	Attack Acc:0.60
Time:28.18	Acc:0.60	
Iteration:300	Prior Loss:0.06	Iden Loss:0.00	Attack Acc:0.80
Time:27.69	Acc:0.80	
Acc:0.73	Acc_5:0.97	Acc_var:0.0133	Acc_var5:0.0033
seed

In [ ]:
# Attack FaceNet64
!python recovery.py \
    --model FaceNet64 \
    --iter_list 300,600,1200 \
    --num_ids 10 \
    --num_seeds 3 \
    --improved_flag \
    --phase phase1

[2026-03-01 18:08:27,317 INFO recovery.py line 68 9970] Namespace(model='FaceNet64', device='4,5,6,7', improved_flag=True, dist_flag=False, iter_list='300,600,1200', num_ids=10, num_seeds=3, phase='phase1', defense='none', noise_sigma=0.01, smooth_alpha=0.1, model_trained_against=None)
[2026-03-01 18:08:27,317 INFO recovery.py line 69 9970] => creating model ...
/content/drive/MyDrive/Knowledge-Enriched-DMI-main/discri.py:15: FutureWarning: `nn.init.normal` is now deprecated in favor of `nn.init.normal_`.
  init.normal(self.T, 0, 1)
[2026-03-01 18:08:35,817 INFO recovery.py line 172 9970] => Running phase1
[2026-03-01 18:08:36,294 INFO recovery.py line 191 9970] ==> Running with iter_times=300
Iteration:300	Prior Loss:0.09	Iden Loss:0.00	Attack Acc:0.60
Time:17.97	Acc:0.60	
Iteration:300	Prior Loss:0.10	Iden Loss:0.00	Attack Acc:0.50
Time:17.69	Acc:0.50	
Iteration:300	Prior Loss:0.06	Iden Loss:0.00	Attack Acc:0.90
Time:17.53	Acc:0.90	
Acc:0.67	Acc_5:0.83	Acc_var:0.0433	Acc_var5:0.0133


## Phase 2 - Different Types of Attacks Comparisions

In [ ]:
# Baseline Attack on VGG16
!python recovery.py \
  --model VGG16 \
  --iter_list 600 \
  --num_ids 30 \
  --num_seeds 5 \
  --phase phase2

[2026-02-28 00:26:02,356 INFO recovery.py line 61 6140] Namespace(model='VGG16', device='4,5,6,7', improved_flag=False, dist_flag=False, iter_list='600', num_ids=30, num_seeds=5, phase='phase2')
[2026-02-28 00:26:02,356 INFO recovery.py line 62 6140] => creating model ...
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_BN_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_BN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
[2026-02-28 00:26:06,397 INFO recovery.py line 136 6140] => Running phase2
[2026-02-28 00:26:06,403 INFO

In [ ]:
# Baseline Attack on IR152
!python recovery.py \
  --model IR152 \
  --iter_list 600 \
  --num_ids 30 \
  --num_seeds 5 \
  --phase phase2

[2026-02-28 01:16:32,662 INFO recovery.py line 61 19566] Namespace(model='IR152', device='4,5,6,7', improved_flag=False, dist_flag=False, iter_list='600', num_ids=30, num_seeds=5, phase='phase2')
[2026-02-28 01:16:32,662 INFO recovery.py line 62 19566] => creating model ...
[2026-02-28 01:16:45,097 INFO recovery.py line 136 19566] => Running phase2
[2026-02-28 01:16:45,102 INFO recovery.py line 220 19566] ==> Running Baseline attack
Iteration:300	Prior Loss:40.64	Iden Loss:0.01	Attack Acc:0.40
Iteration:600	Prior Loss:38.42	Iden Loss:0.01	Attack Acc:0.37
Time:103.78	Acc:0.37	
Iteration:300	Prior Loss:42.66	Iden Loss:0.01	Attack Acc:0.20
Iteration:600	Prior Loss:41.13	Iden Loss:0.01	Attack Acc:0.20
Time:106.39	Acc:0.20	
Iteration:300	Prior Loss:41.69	Iden Loss:0.01	Attack Acc:0.27
Iteration:600	Prior Loss:40.10	Iden Loss:0.01	Attack Acc:0.30
Time:106.80	Acc:0.30	
Iteration:300	Prior Loss:41.39	Iden Loss:0.02	Attack Acc:0.40
Iteration:600	Prior Loss:39.40	Iden Loss:0.01	Attack Acc:0.33
T

In [ ]:
# Baseline attack on FaceNet64
!python recovery.py \
  --model FaceNet64 \
  --iter_list 600 \
  --num_ids 30 \
  --num_seeds 5 \
  --phase phase2

[2026-02-28 01:25:41,985 INFO recovery.py line 61 22003] Namespace(model='FaceNet64', device='4,5,6,7', improved_flag=False, dist_flag=False, iter_list='600', num_ids=30, num_seeds=5, phase='phase2')
[2026-02-28 01:25:41,985 INFO recovery.py line 62 22003] => creating model ...
[2026-02-28 01:25:52,202 INFO recovery.py line 136 22003] => Running phase2
[2026-02-28 01:25:52,207 INFO recovery.py line 220 22003] ==> Running Baseline attack
Iteration:300	Prior Loss:41.25	Iden Loss:0.01	Attack Acc:0.20
Iteration:600	Prior Loss:38.89	Iden Loss:0.01	Attack Acc:0.27
Time:71.46	Acc:0.27	
Iteration:300	Prior Loss:40.61	Iden Loss:0.01	Attack Acc:0.30
Iteration:600	Prior Loss:38.64	Iden Loss:0.01	Attack Acc:0.40
Time:70.24	Acc:0.40	
Iteration:300	Prior Loss:41.40	Iden Loss:0.01	Attack Acc:0.40
Iteration:600	Prior Loss:39.50	Iden Loss:0.00	Attack Acc:0.30
Time:70.26	Acc:0.30	
Iteration:300	Prior Loss:40.68	Iden Loss:0.01	Attack Acc:0.33
Iteration:600	Prior Loss:38.85	Iden Loss:0.01	Attack Acc:0.30


In [ ]:
# Improved GAN attack on VGG16
!python recovery.py \
  --model VGG16 \
  --iter_list 600 \
  --num_ids 30 \
  --num_seeds 5 \
  --improved_flag \
  --phase phase2

[2026-02-28 00:59:29,572 INFO recovery.py line 61 14892] Namespace(model='VGG16', device='4,5,6,7', improved_flag=True, dist_flag=False, iter_list='600', num_ids=30, num_seeds=5, phase='phase2')
[2026-02-28 00:59:29,572 INFO recovery.py line 62 14892] => creating model ...
/content/drive/MyDrive/Knowledge-Enriched-DMI-main/discri.py:15: FutureWarning: `nn.init.normal` is now deprecated in favor of `nn.init.normal_`.
  init.normal(self.T, 0, 1)
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_BN_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_BN_Wei

In [ ]:
# Improved GAN attack on IR152
!python recovery.py \
  --model IR152 \
  --iter_list 600 \
  --num_ids 30 \
  --num_seeds 5 \
  --improved_flag \
  --phase phase2

[2026-02-28 01:31:50,469 INFO recovery.py line 61 23642] Namespace(model='IR152', device='4,5,6,7', improved_flag=True, dist_flag=False, iter_list='600', num_ids=30, num_seeds=5, phase='phase2')
[2026-02-28 01:31:50,469 INFO recovery.py line 62 23642] => creating model ...
/content/drive/MyDrive/Knowledge-Enriched-DMI-main/discri.py:15: FutureWarning: `nn.init.normal` is now deprecated in favor of `nn.init.normal_`.
  init.normal(self.T, 0, 1)
[2026-02-28 01:31:54,372 INFO recovery.py line 136 23642] => Running phase2
[2026-02-28 01:31:54,376 INFO recovery.py line 220 23642] ==> Running Improved_GAN attack
Iteration:300	Prior Loss:0.23	Iden Loss:0.01	Attack Acc:0.63
Iteration:600	Prior Loss:0.14	Iden Loss:0.00	Attack Acc:0.63
Time:106.43	Acc:0.63	
Iteration:300	Prior Loss:0.12	Iden Loss:0.00	Attack Acc:0.43
Iteration:600	Prior Loss:0.07	Iden Loss:0.00	Attack Acc:0.53
Time:105.68	Acc:0.53	
Iteration:300	Prior Loss:0.16	Iden Loss:0.00	Attack Acc:0.50
Iteration:600	Prior Loss:0.09	Iden Lo

In [ ]:
# Improved GAN attack on FaceNet64
!python recovery.py \
  --model FaceNet64 \
  --iter_list 600 \
  --num_ids 30 \
  --num_seeds 5 \
  --improved_flag \
  --phase phase2

[2026-02-28 01:40:48,983 INFO recovery.py line 61 26024] Namespace(model='FaceNet64', device='4,5,6,7', improved_flag=True, dist_flag=False, iter_list='600', num_ids=30, num_seeds=5, phase='phase2')
[2026-02-28 01:40:48,983 INFO recovery.py line 62 26024] => creating model ...
/content/drive/MyDrive/Knowledge-Enriched-DMI-main/discri.py:15: FutureWarning: `nn.init.normal` is now deprecated in favor of `nn.init.normal_`.
  init.normal(self.T, 0, 1)
[2026-02-28 01:40:52,299 INFO recovery.py line 136 26024] => Running phase2
[2026-02-28 01:40:52,303 INFO recovery.py line 220 26024] ==> Running Improved_GAN attack
Iteration:300	Prior Loss:0.18	Iden Loss:0.00	Attack Acc:0.43
Iteration:600	Prior Loss:0.08	Iden Loss:0.00	Attack Acc:0.47
Time:69.86	Acc:0.47	
Iteration:300	Prior Loss:0.12	Iden Loss:0.00	Attack Acc:0.53
Iteration:600	Prior Loss:0.06	Iden Loss:0.00	Attack Acc:0.53
Time:68.99	Acc:0.53	
Iteration:300	Prior Loss:0.17	Iden Loss:0.00	Attack Acc:0.43
Iteration:600	Prior Loss:0.11	Iden 

In [ ]:
# Distributional attack on VGG16
!python recovery.py \
  --model VGG16 \
  --iter_list 600 \
  --num_ids 30 \
  --num_seeds 5 \
  --improved_flag \
  --dist_flag \
  --phase phase2

[2026-02-28 01:12:42,104 INFO recovery.py line 61 18470] Namespace(model='VGG16', device='4,5,6,7', improved_flag=True, dist_flag=True, iter_list='600', num_ids=30, num_seeds=5, phase='phase2')
[2026-02-28 01:12:42,104 INFO recovery.py line 62 18470] => creating model ...
/content/drive/MyDrive/Knowledge-Enriched-DMI-main/discri.py:15: FutureWarning: `nn.init.normal` is now deprecated in favor of `nn.init.normal_`.
  init.normal(self.T, 0, 1)
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_BN_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_BN_Weig

In [ ]:
# Distributional attack on IR152
!python recovery.py \
  --model IR152 \
  --iter_list 600 \
  --num_ids 30 \
  --num_seeds 5 \
  --improved_flag \
  --dist_flag \
  --phase phase2

[2026-02-28 01:46:43,811 INFO recovery.py line 61 27599] Namespace(model='IR152', device='4,5,6,7', improved_flag=True, dist_flag=True, iter_list='600', num_ids=30, num_seeds=5, phase='phase2')
[2026-02-28 01:46:43,811 INFO recovery.py line 62 27599] => creating model ...
/content/drive/MyDrive/Knowledge-Enriched-DMI-main/discri.py:15: FutureWarning: `nn.init.normal` is now deprecated in favor of `nn.init.normal_`.
  init.normal(self.T, 0, 1)
[2026-02-28 01:46:48,206 INFO recovery.py line 136 27599] => Running phase2
[2026-02-28 01:46:48,210 INFO recovery.py line 220 27599] ==> Running Distributional_Recovery attack
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:80: FutureWarning: Using -1 to represent CPU tensor is deprecated. Please use a device object or string instead, e.g., "cpu".
  return comm.gather(inputs, ctx.dim, ctx.target_device)
Iteration:300	Prior Loss:5.15	Iden Loss:1.43	Attack Acc:0.23
Iteration:600	Prior Loss:4.58	Iden Loss:0.34	Attack Acc:0.40

In [ ]:
# Distributional attack on FaceNet64
!python recovery.py \
  --model FaceNet64 \
  --iter_list 600 \
  --num_ids 30 \
  --num_seeds 5 \
  --improved_flag \
  --dist_flag \
  --phase phase2

[2026-02-28 01:48:43,708 INFO recovery.py line 61 28121] Namespace(model='FaceNet64', device='4,5,6,7', improved_flag=True, dist_flag=True, iter_list='600', num_ids=30, num_seeds=5, phase='phase2')
[2026-02-28 01:48:43,708 INFO recovery.py line 62 28121] => creating model ...
/content/drive/MyDrive/Knowledge-Enriched-DMI-main/discri.py:15: FutureWarning: `nn.init.normal` is now deprecated in favor of `nn.init.normal_`.
  init.normal(self.T, 0, 1)
[2026-02-28 01:48:46,305 INFO recovery.py line 136 28121] => Running phase2
[2026-02-28 01:48:46,313 INFO recovery.py line 220 28121] ==> Running Distributional_Recovery attack
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:80: FutureWarning: Using -1 to represent CPU tensor is deprecated. Please use a device object or string instead, e.g., "cpu".
  return comm.gather(inputs, ctx.dim, ctx.target_device)
Iteration:300	Prior Loss:4.70	Iden Loss:1.72	Attack Acc:0.30
Iteration:600	Prior Loss:4.63	Iden Loss:0.27	Attack Acc:

## Phase 3: Cross-Model Transfer Attacks (and Defenses)

### White-box and transfer attacks (without defenses)

In [ ]:
# White-Box Attack on CGG16
!python recovery.py \
--model VGG16 \
--model_trained_against VGG16 \
--improved_flag \
--iter_list 600 \
--num_ids 20 \
--num_seeds 5 \
--phase phase3

[2026-03-01 13:20:23,075 INFO recovery.py line 68 8303] Namespace(model='VGG16', device='4,5,6,7', improved_flag=True, dist_flag=False, iter_list='600', num_ids=20, num_seeds=5, phase='phase3', defense='none', noise_sigma=0.01, smooth_alpha=0.1, model_trained_against='VGG16')
[2026-03-01 13:20:23,075 INFO recovery.py line 69 8303] => creating model ...
/content/drive/MyDrive/Knowledge-Enriched-DMI-main/discri.py:15: FutureWarning: `nn.init.normal` is now deprecated in favor of `nn.init.normal_`.
  init.normal(self.T, 0, 1)
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing

In [ ]:
# Transfer Attack on IR152 using model trained against VGG16
!python recovery.py \
--model IR152 \
--model_trained_against VGG16 \
--improved_flag \
--iter_list 600 \
--num_ids 20 \
--num_seeds 5 \
--phase phase3

[2026-03-01 13:23:47,255 INFO recovery.py line 68 9247] Namespace(model='IR152', device='4,5,6,7', improved_flag=True, dist_flag=False, iter_list='600', num_ids=20, num_seeds=5, phase='phase3', defense='none', noise_sigma=0.01, smooth_alpha=0.1, model_trained_against='VGG16')
[2026-03-01 13:23:47,256 INFO recovery.py line 69 9247] => creating model ...
/content/drive/MyDrive/Knowledge-Enriched-DMI-main/discri.py:15: FutureWarning: `nn.init.normal` is now deprecated in favor of `nn.init.normal_`.
  init.normal(self.T, 0, 1)
[2026-03-01 13:23:59,871 INFO recovery.py line 172 9247] => Running phase3
[2026-03-01 13:23:59,873 INFO recovery.py line 317 9247] ==> Phase3: Improved_GAN
Iteration:300	Prior Loss:0.17	Iden Loss:0.00	Attack Acc:0.50
Iteration:600	Prior Loss:0.08	Iden Loss:0.00	Attack Acc:0.50
Time:89.07	Acc:0.50	
Iteration:300	Prior Loss:0.14	Iden Loss:0.00	Attack Acc:0.65
Iteration:600	Prior Loss:0.08	Iden Loss:0.00	Attack Acc:0.65
Time:88.28	Acc:0.65	
Iteration:300	Prior Loss:0.1

In [ ]:
# Transfer Attack on FaceNet64 using model trained against VGG16
!python recovery.py \
--model FaceNet64 \
--model_trained_against VGG16 \
--improved_flag \
--iter_list 600 \
--num_ids 20 \
--num_seeds 5 \
--phase phase3

[2026-03-01 13:31:27,811 INFO recovery.py line 68 11156] Namespace(model='FaceNet64', device='4,5,6,7', improved_flag=True, dist_flag=False, iter_list='600', num_ids=20, num_seeds=5, phase='phase3', defense='none', noise_sigma=0.01, smooth_alpha=0.1, model_trained_against='VGG16')
[2026-03-01 13:31:27,811 INFO recovery.py line 69 11156] => creating model ...
/content/drive/MyDrive/Knowledge-Enriched-DMI-main/discri.py:15: FutureWarning: `nn.init.normal` is now deprecated in favor of `nn.init.normal_`.
  init.normal(self.T, 0, 1)
[2026-03-01 13:31:37,874 INFO recovery.py line 172 11156] => Running phase3
[2026-03-01 13:31:37,875 INFO recovery.py line 317 11156] ==> Phase3: Improved_GAN
Iteration:300	Prior Loss:0.10	Iden Loss:0.00	Attack Acc:0.55
Iteration:600	Prior Loss:0.05	Iden Loss:0.00	Attack Acc:0.55
Time:57.82	Acc:0.55	
Iteration:300	Prior Loss:0.13	Iden Loss:0.00	Attack Acc:0.45
Iteration:600	Prior Loss:0.06	Iden Loss:0.00	Attack Acc:0.45
Time:55.24	Acc:0.45	
Iteration:300	Prior 

In [ ]:
# White-box Attack IR152
!python recovery.py \
--model IR152 \
--model_trained_against IR152 \
--improved_flag \
--iter_list 600 \
--num_ids 20 \
--num_seeds 5 \
--phase phase3

[2026-03-01 14:05:14,253 INFO recovery.py line 68 19864] Namespace(model='IR152', device='4,5,6,7', improved_flag=True, dist_flag=False, iter_list='600', num_ids=20, num_seeds=5, phase='phase3', defense='none', noise_sigma=0.01, smooth_alpha=0.1, model_trained_against='IR152')
[2026-03-01 14:05:14,253 INFO recovery.py line 69 19864] => creating model ...
/content/drive/MyDrive/Knowledge-Enriched-DMI-main/discri.py:15: FutureWarning: `nn.init.normal` is now deprecated in favor of `nn.init.normal_`.
  init.normal(self.T, 0, 1)
[2026-03-01 14:05:17,737 INFO recovery.py line 172 19864] => Running phase3
[2026-03-01 14:05:17,738 INFO recovery.py line 317 19864] ==> Phase3: Improved_GAN
Iteration:300	Prior Loss:0.15	Iden Loss:0.01	Attack Acc:0.65
Iteration:600	Prior Loss:0.08	Iden Loss:0.00	Attack Acc:0.65
Time:89.34	Acc:0.65	
Iteration:300	Prior Loss:0.12	Iden Loss:0.00	Attack Acc:0.50
Iteration:600	Prior Loss:0.08	Iden Loss:0.00	Attack Acc:0.60
Time:88.27	Acc:0.60	
Iteration:300	Prior Loss

### White-box and Transfer Attacks with Defenses

In [ ]:
# White-box Attack on VGG16
# Using Defense - Gaussian Noise
!python recovery.py \
--model VGG16 \
--model_trained_against VGG16 \
--improved_flag \
--iter_list 600 \
--num_ids 20 \
--num_seeds 3 \
--phase phase3 \
--defense noise \
--noise_sigma 0.01

[2026-03-01 18:14:46,300 INFO recovery.py line 68 11656] Namespace(model='VGG16', device='4,5,6,7', improved_flag=True, dist_flag=False, iter_list='600', num_ids=20, num_seeds=3, phase='phase3', defense='noise', noise_sigma=0.01, smooth_alpha=0.1, model_trained_against='VGG16')
[2026-03-01 18:14:46,300 INFO recovery.py line 69 11656] => creating model ...
/content/drive/MyDrive/Knowledge-Enriched-DMI-main/discri.py:15: FutureWarning: `nn.init.normal` is now deprecated in favor of `nn.init.normal_`.
  init.normal(self.T, 0, 1)
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to pass

In [ ]:
# White-box Attack on VGG16
# Using Defense - Label Smoothing
!python recovery.py \
--model VGG16 \
--model_trained_against VGG16 \
--improved_flag \
--iter_list 600 \
--num_ids 20 \
--num_seeds 3 \
--phase phase3 \
--defense smooth \
--smooth_alpha 0.1

[2026-03-01 18:21:06,480 INFO recovery.py line 68 13359] Namespace(model='VGG16', device='4,5,6,7', improved_flag=True, dist_flag=False, iter_list='600', num_ids=20, num_seeds=3, phase='phase3', defense='smooth', noise_sigma=0.01, smooth_alpha=0.1, model_trained_against='VGG16')
[2026-03-01 18:21:06,480 INFO recovery.py line 69 13359] => creating model ...
/content/drive/MyDrive/Knowledge-Enriched-DMI-main/discri.py:15: FutureWarning: `nn.init.normal` is now deprecated in favor of `nn.init.normal_`.
  init.normal(self.T, 0, 1)
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to pas

In [ ]:
# Transfer Attack on IR152 using model trained against VGG16
# Using Defense - Gaussian Noise
!python recovery.py \
--model IR152 \
--model_trained_against VGG16 \
--improved_flag \
--iter_list 600 \
--num_ids 20 \
--num_seeds 3 \
--phase phase3 \
--defense noise \
--noise_sigma 0.01

[2026-03-01 18:16:38,393 INFO recovery.py line 68 12164] Namespace(model='IR152', device='4,5,6,7', improved_flag=True, dist_flag=False, iter_list='600', num_ids=20, num_seeds=3, phase='phase3', defense='noise', noise_sigma=0.01, smooth_alpha=0.1, model_trained_against='VGG16')
[2026-03-01 18:16:38,393 INFO recovery.py line 69 12164] => creating model ...
/content/drive/MyDrive/Knowledge-Enriched-DMI-main/discri.py:15: FutureWarning: `nn.init.normal` is now deprecated in favor of `nn.init.normal_`.
  init.normal(self.T, 0, 1)
[2026-03-01 18:16:42,221 INFO recovery.py line 172 12164] => Running phase3
[2026-03-01 18:16:42,222 INFO recovery.py line 317 12164] ==> Phase3: Improved_GAN
Iteration:300	Prior Loss:0.20	Iden Loss:0.01	Attack Acc:0.65
Iteration:600	Prior Loss:0.12	Iden Loss:0.00	Attack Acc:0.70
Time:86.81	Acc:0.70	
Iteration:300	Prior Loss:0.15	Iden Loss:0.00	Attack Acc:0.65
Iteration:600	Prior Loss:0.09	Iden Loss:0.00	Attack Acc:0.65
Time:85.83	Acc:0.65	
Iteration:300	Prior Los

In [ ]:
# Transfer Attack on IR152 using model trained against VGG16
# Using Defense - Label Smoothing
!python recovery.py \
--model IR152 \
--model_trained_against VGG16 \
--improved_flag \
--iter_list 600 \
--num_ids 20 \
--num_seeds 3 \
--phase phase3 \
--defense smooth \
--smooth_alpha 0.1

[2026-03-01 18:22:56,745 INFO recovery.py line 68 13861] Namespace(model='IR152', device='4,5,6,7', improved_flag=True, dist_flag=False, iter_list='600', num_ids=20, num_seeds=3, phase='phase3', defense='smooth', noise_sigma=0.01, smooth_alpha=0.1, model_trained_against='VGG16')
[2026-03-01 18:22:56,745 INFO recovery.py line 69 13861] => creating model ...
/content/drive/MyDrive/Knowledge-Enriched-DMI-main/discri.py:15: FutureWarning: `nn.init.normal` is now deprecated in favor of `nn.init.normal_`.
  init.normal(self.T, 0, 1)
[2026-03-01 18:22:59,967 INFO recovery.py line 172 13861] => Running phase3
[2026-03-01 18:22:59,968 INFO recovery.py line 317 13861] ==> Phase3: Improved_GAN
Iteration:300	Prior Loss:0.15	Iden Loss:0.11	Attack Acc:0.50
Iteration:600	Prior Loss:0.08	Iden Loss:0.11	Attack Acc:0.55
Time:86.70	Acc:0.55	
Iteration:300	Prior Loss:0.31	Iden Loss:0.11	Attack Acc:0.55
Iteration:600	Prior Loss:0.18	Iden Loss:0.11	Attack Acc:0.60
Time:85.78	Acc:0.60	
Iteration:300	Prior Lo

### Post-run Logs Processing
- While running phase3 attacks, I did not save seed-level accuracies (only the average accuracy was saved)
- While running statistical tests later (link to notebook -), I realized that seed-level accuracies are needed to gather any sort of statistical insights
- Wrote some code to read seed-level accuracies and the train/test model names from phase3 output logs above and save them (given the limited compute, I could not modify the code and re-run all the attacks again)
- If you wish to run it on your system, please paste each phase3 cell output one by one below in the colab_output variable


In [ ]:
def parse_colab_to_csv(text):
    data = []

    blocks = text.split("Namespace")

    for block in blocks[1:]:
        model_match = re.search(r"model='([^']*)'", block)
        trained_match = re.search(r"model_trained_against='([^']*)'", block)
        defense_match = re.search(r"defense='([^']*)'", block)

        if model_match and trained_match:
            test_model = model_match.group(1)
            train_model = trained_match.group(1)
            defense = defense_match.group(1) if defense_match else "none"

            seed_accs = re.findall(r"Time:[\d.]+\s+Acc:([\d.]+)", block)

            for acc in seed_accs:
                data.append({
                    "Test_Model": test_model,
                    "Model_Trained_Against": train_model,
                    "Defense": defense,
                    "Top1_Acc": float(acc)
                })

    return pd.DataFrame(data)

In [20]:
# Paste your Colab output between the quotes below
colab_output = """
[2026-03-01 14:05:14,253 INFO recovery.py line 68 19864] Namespace(model='IR152', device='4,5,6,7', improved_flag=True, dist_flag=False, iter_list='600', num_ids=20, num_seeds=5, phase='phase3', defense='none', noise_sigma=0.01, smooth_alpha=0.1, model_trained_against='IR152')
[2026-03-01 14:05:14,253 INFO recovery.py line 69 19864] => creating model ...
/content/drive/MyDrive/Knowledge-Enriched-DMI-main/discri.py:15: FutureWarning: `nn.init.normal` is now deprecated in favor of `nn.init.normal_`.
  init.normal(self.T, 0, 1)
[2026-03-01 14:05:17,737 INFO recovery.py line 172 19864] => Running phase3
[2026-03-01 14:05:17,738 INFO recovery.py line 317 19864] ==> Phase3: Improved_GAN
Iteration:300	Prior Loss:0.15	Iden Loss:0.01	Attack Acc:0.65
Iteration:600	Prior Loss:0.08	Iden Loss:0.00	Attack Acc:0.65
Time:89.34	Acc:0.65
Iteration:300	Prior Loss:0.12	Iden Loss:0.00	Attack Acc:0.50
Iteration:600	Prior Loss:0.08	Iden Loss:0.00	Attack Acc:0.60
Time:88.27	Acc:0.60
Iteration:300	Prior Loss:0.20	Iden Loss:0.01	Attack Acc:0.60
Iteration:600	Prior Loss:0.12	Iden Loss:0.00	Attack Acc:0.55
Time:88.28	Acc:0.55
Iteration:300	Prior Loss:0.13	Iden Loss:0.01	Attack Acc:0.50
Iteration:600	Prior Loss:0.08	Iden Loss:0.00	Attack Acc:0.50
Time:88.19	Acc:0.50
Iteration:300	Prior Loss:0.14	Iden Loss:0.00	Attack Acc:0.35
Iteration:600	Prior Loss:0.07	Iden Loss:0.00	Attack Acc:0.45
Time:87.98	Acc:0.45
Acc:0.55	Acc_5:0.78	Acc_var:0.0063	Acc_var5:0.0020
seeds variance: tensor([0.3000, 0.2000, 0.3000, 0.2000, 0.2000, 0.2000, 0.3000, 0.2000, 0.2000,
        0.3000, 0.0000, 0.3000, 0.3000, 0.0000, 0.2000, 0.2000, 0.3000, 0.2000,
        0.2000, 0.0000])
Results for Improved_GAN attack with defense=none, noise_sigma=0.01, smooth_alpha=0.1: Acc=0.5500, Acc5=0.7800, Var=0.006250, Var5=0.002000, Runtime=442.17 sec
[2026-03-01 14:12:39,908 INFO recovery.py line 364 19864] Results saved to ./results_phase3/results.csv
"""

df_seeds = parse_colab_to_csv(colab_output)

file_path = ROOT / "results_phase3" / "seed_level_results.csv"
file_exists = os.path.isfile(file_path)
df_seeds.to_csv(file_path, mode='a', index=False, header=not file_exists)

print(df_seeds.head(10))

  Test_Model Model_Trained_Against Defense  Top1_Acc
0      IR152                 IR152    none      0.65
1      IR152                 IR152    none      0.60
2      IR152                 IR152    none      0.55
3      IR152                 IR152    none      0.50
4      IR152                 IR152    none      0.45
